01_eda_tess.ipynb (EDA pre TESS)
- Cieľ notebooku
- ukázať rozdelenie tried (stress/no-stress),
- overiť tvary .npy features a spektrogramov,
- zobraziť pár ukážok spektrogramov,
základné štatistiky (dĺžky, min/max/mean).

In [ ]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = os.path.abspath("..")  # ak notebook beží v projekt/notebooks
TESS_PROC = os.path.join(PROJECT_ROOT, "data", "processed", "tess")

FEAT_DIR = os.path.join(TESS_PROC, "features")
SPEC_DIR = os.path.join(TESS_PROC, "spectrograms")

feat_files = sorted(glob.glob(os.path.join(FEAT_DIR, "*.npy")))
spec_files = sorted(glob.glob(os.path.join(SPEC_DIR, "*.npy")))

len(feat_files), len(spec_files), feat_files[:3]

In [ ]:
# Mapovanie presne ako v config.py
TESS_LABEL_MAPPING = {
    "neutral": 0,
    "calm": 0,
    "angry": 1,
    "fear": 1,
    "disgust": 1,
    "ps": 0,
    "sad": 1,
}

KNOWN_EMOTIONS = set(TESS_LABEL_MAPPING.keys())

def parse_emotion_tess(path: str) -> str:
    stem = os.path.basename(path).replace(".npy", "")
    parts = stem.split("_")
    # v tvojich názvoch je emočný token napr. angry/neutral/ps/sad...
    emo = None
    for p in parts:
        if p in KNOWN_EMOTIONS:
            emo = p
            break
    if emo is None:
        raise ValueError(f"Neviem nájsť emóciu v názve: {path}")
    return emo

def emotion_to_label(emo: str) -> int:
    return TESS_LABEL_MAPPING[emo]

emotions = [parse_emotion_tess(f) for f in feat_files]
labels = [emotion_to_label(e) for e in emotions]

pd.Series(emotions).value_counts().head(10), pd.Series(labels).value_counts().sort_index()

In [ ]:
x = np.load(feat_files[0])
print("Example feature shape:", x.shape, x.dtype)

# ak sú features 2D (D, T), pozri distribúciu T
Ts = []
Ds = []
for f in feat_files[:300]:
    arr = np.load(f)
    if arr.ndim == 2:
        Ds.append(arr.shape[0])
        Ts.append(arr.shape[1])
pd.Series(Ds).value_counts(), pd.Series(Ts).describe()

In [ ]:
if len(spec_files) > 0:
    spec = np.load(spec_files[0])
    plt.figure(figsize=(10,4))
    plt.imshow(spec, aspect="auto", origin="lower")
    plt.colorbar()
    plt.title(os.path.basename(spec_files[0]))
    plt.xlabel("time")
    plt.ylabel("mel bins")
    plt.show()
else:
    print("spectrograms/ nie sú dostupné")

In [ ]:
counts = pd.Series(labels).value_counts().sort_index()
plt.figure(figsize=(4,3))
counts.plot(kind="bar")
plt.title("TESS: counts per label (0=no-stress, 1=stress)")
plt.xlabel("label")
plt.ylabel("count")
plt.grid(axis="y", alpha=0.3)
plt.show()